In [1]:
import os
import sys
from pathlib import Path

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

CODE_DIR: /home/daniel/HRM_Reconstruction/code
CODE_DIR contents: ['.GPT2_Model', 'Utils', 'HRM_Model', 'Datasets', '.DS_Store', 'BiLSTM_Model', 'Sudoku', 'BERT_Model']
Device set to cuda


In [ ]:
from Datasets.Sudoku_DataLoader import get_loaders

from BiLSTM_Model.BiLSTM_Sudoku import SudokuBiLSTM, BiLSTMConfig
from BiLSTM_Model.BiLSTM_Train import train_bilstm

from Datasets.Sudoku_DataLoader import collect_puzzles_set
from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint
from Utils.visualization import show_sudoku_predictions, print_sudoku_comparison

/home/daniel/anaconda3/envs/HRM/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_size = 2**18
test_size = 2**15
batch_size = 2**7

train_loader, val_loader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

Map: 100%|██████████| 32768/32768 [00:00<00:00, 39121.14 examples/s]


In [ ]:
print("Collecting train puzzles...")
train_puzzles = collect_puzzles_set(train_loader)

print("Collecting val puzzles...")
val_puzzles = collect_puzzles_set(val_loader)

overlap = train_puzzles.intersection(val_puzzles)

print(f"\nTrain puzzles: {len(train_puzzles)}")
print(f"Val puzzles:   {len(val_puzzles)}")
print(f"Overlap:       {len(overlap)}")

if len(overlap) > 0:
    print("WARNING: Puzzle overlap detected!")
else:
    print("No puzzle overlap between train and validation sets.")

In [4]:
model_config = BiLSTMConfig(
    vocab_size=10,
    num_classes=10,
    embedding_dim=512,
    hidden_size=560,
    num_layers=5,
    dropout=0.2
)

lr = 1e-4
min_lr_ratio = 0.1 # -> 1e-5
lr_warmup = 0.05
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 8 * 20 # since HRM M=8, this gives same num_training_steps

checkpoint_dir = "checkpoints"

In [5]:
model = SudokuBiLSTM(model_config).to(device)

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

Number of trainable parameters: 34,969,290


In [7]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(lr_warmup * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=min_lr_ratio
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

num_training_steps: 327680
num_warmup_steps: 16384


In [ ]:
model, best_metric, history = train_bilstm(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=5,
    validate_every=5,
    val_loader=val_loader,
    grad_clip=None,
    step_val_batches=1,
    step_val_every=8,
)
model.eval()

print("Best board accuracy used for checkpointing:", best_metric)

Number of trainable parameters: 34,969,290


Epoch 1: 100%|██████████| 2048/2048 [02:45<00:00, 12.37it/s]


Epoch 1: Avg Train Loss = 2.1359, Train Token Accuracy = 14.61%, Train Board Accuracy = 0.00%, LR = 1.25e-05


Epoch 2: 100%|██████████| 2048/2048 [02:44<00:00, 12.44it/s]


Epoch 2: Avg Train Loss = 2.0109, Train Token Accuracy = 16.47%, Train Board Accuracy = 0.00%, LR = 2.50e-05


...

Epoch 107: 100%|██████████| 2048/2048 [02:48<00:00, 12.12it/s]


Epoch 107: Avg Train Loss = 0.3134, Train Token Accuracy = 88.24%, Train Board Accuracy = 2.53%, LR = 3.44e-05


Epoch 108: 100%|██████████| 2048/2048 [02:49<00:00, 12.09it/s]


Epoch 108: Avg Train Loss = 0.3086, Train Token Accuracy = 88.42%, Train Board Accuracy = 2.67%, LR = 3.36e-05


Epoch 109:  22%|██▏       | 441/2048 [00:35<02:16, 11.76it/s]

Same strange loss behavior as BERT, except Val Board Accuracy doesn't get as high

In [ ]:
import matplotlib.pyplot as plt

train_steps = history["step"]
train_loss = history["train_loss"]

val_steps = [
    s for s, v in zip(history["step"], history["val_loss"])
    if v is not None
]

val_loss = [
    v for v in history["val_loss"]
    if v is not None
]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Train loss", linewidth=1)

if len(val_loss) > 0:
    plt.plot(val_steps, val_loss, label="Validation (subset)", linewidth=2)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("BiLSTM Loss vs Training Step")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
show_sudoku_predictions(
    model,
    val_loader,
    device,
)